# Assignment7

そのまま課題ファイルで編集しないでください。課題ファイルを複製してから課題を回答してください。

## 課題1

pandasでDataフォルダの```example.json```を読み込む。

In [7]:
import pandas as pd

df_json = pd.read_json('example.json', lines=True)
df_json

,Name,Price,Model,Power
0,Honda,10000,2005,1300
1,Toyota,12000,2010,1600
2,Audi,25000,2017,1800
3,Ford,28000,2009,1200


## 課題2

pandasでDataフォルダの```state-abbrevs.csv```、```state-areas.csv```、```state-population.csv```を読み込む。

これらのデータを使って、**2010年**アメリカの各州の人口密度を計算しよう。

In [8]:
# CSVファイルの読み込み
pop = pd.read_csv('state-population.csv')
areas = pd.read_csv('state-areas.csv')
abbrevs = pd.read_csv('state-abbrevs.csv')

# 1. 人口データと州略称データを結合
merged = pd.merge(pop, abbrevs, how='outer', left_on='state/region', right_on='abbreviation')
merged = merged.drop('abbreviation', axis=1)

# PRやUSAなど補完
merged.loc[merged['state/region'] == 'PR', 'state'] = 'Puerto Rico'
merged.loc[merged['state/region'] == 'USA', 'state'] = 'United States'

# 2. 面積データを結合
final = pd.merge(merged, areas, on='state', how='left')

# 3. 2010年かつ全年齢（total）のデータに絞り込み
data2010 = final.query("year == 2010 & ages == 'total'").copy()

# 4. 人口密度を計算
data2010['density'] = data2010['population'] / data2010['area (sq. mi)']

# 5. 人口密度でソートして表示
data2010.sort_values('density', ascending=False, inplace=True)
data2010.head()

,state/region,ages,year,population,state,area (sq. mi),density
341,DC,total,2010,605125.0,District of Columbia,68.0,8898.897059
1914,PR,total,2010,3721208.0,Puerto Rico,3515.0,1058.665149
1493,NJ,total,2010,8802707.0,New Jersey,8722.0,1009.253268
1962,RI,total,2010,1052669.0,Rhode Island,1545.0,681.339159
293,CT,total,2010,3579210.0,Connecticut,5544.0,645.600649


## 課題3

```state/region```、```ages```と```years```でデータフレームをグループして、各グループで$\frac{population}{area (sq. mi)}$の形で人口密度を計算してください。

In [9]:
# 課題2の結合データ final から、面積が取得できているデータを抽出
df_grouped = final.dropna(subset=['area (sq. mi)']).copy()

# state/region, ages, year でグループ化して人口密度を計算
df_grouped['population_density'] = df_grouped['population'] / df_grouped['area (sq. mi)']

# 指定されたキーでグループ化した結果を表示
result = df_grouped.groupby(['state/region', 'ages', 'year'])[['population', 'area (sq. mi)', 'population_density']].first()
result

population  area (sq. mi)  population_density
state/region ages    year                                               
AK           total   1990    553290.0       656425.0            0.842884
                     1991    570193.0       656425.0            0.868634
                     1992    588736.0       656425.0            0.896882
                     1993    599434.0       656425.0            0.913180
                     1994    603308.0       656425.0            0.919081
...                               ...            ...                 ...
WY           under18 2009    134960.0        97818.0            1.379705
                     2010    135351.0        97818.0            1.383702
                     2011    135407.0        97818.0            1.384275
                     2012    136526.0        97818.0            1.395714
                     2013    137679.0        97818.0            1.407502

[2496 rows x 3 columns]